# EDA (Google Colab) — ваши данные

Весь код в ноутбуке, **без** `import` из папок репозитория. Сначала скачивается выгрузка `3_5.xlsx`. Графики ВКР строятся **matplotlib + seaborn**, сохраняются как **`eda_output/*.png`** (без HTML).

In [ ]:
!wget -O 3_5.xlsx "https://www.dropbox.com/scl/fi/d4v34uufwttuo7mnjemzi/3_5.xlsx?rlkey=q2ffagpbzp2tfke12va02o6vn&st=2c4be7dl&dl=0"


In [ ]:
%pip install -q pandas numpy pyarrow openpyxl seaborn matplotlib scipy scikit-learn lightgbm


In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, TimeSeriesSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

OUTPUT_DIR = Path("eda_output")
sns.set_palette("deep")


In [ ]:
TARGET_COL = "default_target"
DEFAULT_DPD_THRESHOLD = 90
PDN_DEFAULT_THRESHOLD = 80.0
QUALITY_BAD = ("IV", "V", "4", "5")
RANDOM_STATE = 42
TEST_SIZE = 0.2
MAX_TRAIN_ROWS = 120_000

DATE_COLS = [
    "report_date_as_of", "date_borrower_birthday", "date_return", "date_issued",
    "date_signing", "last_payment_date",
]
NUMERIC_COLS = [
    "initial_amount", "total_debt", "available_limit", "reserve_rate",
    "expired_actual_days_count", "percent_expired_actual_days_count", "dpd",
    "card_commission_expired_days", "limit_conversion_ratio", "total_debt_reserve",
    "available_limit_reserve", "pdn_initial", "pdn_current", "psk_current",
    "ifrs_provision_rate", "total_debt_provision", "available_limit_provision",
    "payment_sum_1m", "payment_sum_2m", "payment_sum_3m", "cnt_all_payments",
]

EWS_ZONES = {
    "green": {"score_max": 0.15},
    "yellow": {"score_min": 0.15, "score_max": 0.40},
    "red": {"score_min": 0.40},
}
EWS_RULES = [
    {"name": "high_dpd", "expr": "dpd > 30", "weight": 2},
    {"name": "bad_quality", "expr": "quality_category.isin(['III','IV','V','3','4','5'])", "weight": 1},
    {"name": "pdn_spike", "expr": "pdn_current > 60", "weight": 1},
    {"name": "no_recent_payment", "expr": "days_since_last_payment > 60", "weight": 2},
    {"name": "payments_drop", "expr": "payment_ratio_mom < 0.3", "weight": 1},
    {"name": "utilization_high", "expr": "utilization > 0.9", "weight": 1},
    {"name": "bankruptcy_trigger", "expr": "is_bankrupt == 1", "weight": 3},
]

SEGMENT_ORDER = ["stable", "growing", "stress", "delinquent"]
ZONE_DISPLAY = {"green": "Зелёная", "yellow": "Жёлтая", "red": "Красная"}
ZONE_COLORS = {"green": "#7fbf7f", "yellow": "#f0c040", "red": "#e06060"}


def _vectorized_eval(expr: str, df: pd.DataFrame) -> np.ndarray:
    env = {c: df[c] for c in df.columns}
    env["__builtins__"] = {}
    env["pd"] = pd
    env["np"] = np
    try:
        result = eval(expr, env)
    except Exception:
        return np.zeros(len(df), dtype=bool)
    if isinstance(result, pd.Series):
        return result.fillna(False).astype(bool).to_numpy()
    if isinstance(result, np.ndarray):
        return result.astype(bool)
    return np.full(len(df), bool(result), dtype=bool)


def apply_rules(df: pd.DataFrame, rules: list) -> pd.DataFrame:
    out = df.copy()
    n = len(out)
    weight_sum = np.zeros(n, dtype=np.float32)
    hit_matrix = np.zeros((n, len(rules)), dtype=np.int8)
    for j, rule in enumerate(rules):
        mask = _vectorized_eval(rule["expr"], out)
        out[f"rule_{rule['name']}"] = mask.astype(np.int8)
        hit_matrix[:, j] = mask.astype(np.int8)
        weight_sum += mask.astype(np.float32) * float(rule.get("weight", 1))
    out["rules_weight_sum"] = weight_sum
    rule_names = [r["name"] for r in rules]
    rule_arr = np.array(rule_names, dtype=object)
    out["rules_triggered"] = [[str(x) for x in rule_arr[row == 1]] for row in hit_matrix]
    return out


def assign_zone(score: float, zones: dict = None) -> str:
    z = zones or EWS_ZONES
    if score < z["green"]["score_max"]:
        return "green"
    if score < z["yellow"]["score_max"]:
        return "yellow"
    return "red"


def mpl_save_show(fig, stem: str) -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(OUTPUT_DIR / f"{stem}.png", dpi=150, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_portfolio_kpi(k: dict, stem: str = "portfolio_kpi") -> None:
    fig, axes = plt.subplots(1, 3, figsize=(12.8, 4.3))
    nb = k.get("n_borrowers", float("nan"))
    y_b = float(nb) if pd.notna(nb) else 0.0
    txt_nb = f"{int(round(y_b)):,}".replace(",", " ") if pd.notna(nb) else "0"
    axes[0].bar(
        ["Договоры", "Заёмщики"],
        [k["n_contracts"], y_b],
        color=["#457b9d", "#a8dadc"],
    )
    axes[0].set_title("Объём портфеля")
    for i, (v, t) in enumerate(zip([k["n_contracts"], y_b], [f'{k["n_contracts"]:,}'.replace(",", " "), txt_nb])):
        axes[0].text(i, v, t, ha="center", va="bottom", fontsize=9)
    debt_bn = k["total_debt"] / 1e9
    lim_bn = k["available_limit"] / 1e9
    res_bn = float(k.get("reserve_amount") or 0) / 1e9
    x2 = ["Задолженность", "Лимит", "Резервы"]
    y2 = [debt_bn, lim_bn, res_bn]
    axes[1].bar(x2, y2, color="#264653")
    axes[1].set_title("Суммы, млрд руб.")
    for i, v in enumerate(y2):
        axes[1].text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=9)
    pdn = k["avg_pdn_current"] * 100
    dpd = k["avg_dpd"]
    dr = k["default_rate"] * 100
    x3 = ["ПДН, %", "DPD, дн.", "Дефолт, %"]
    y3 = [pdn, dpd, dr]
    axes[2].bar(x3, y3, color="#e76f51")
    axes[2].set_title("Риск-показатели")
    for i, v in enumerate(y3):
        axes[2].text(i, v, f"{v:.1f}" if i != 2 else f"{v:.2f}", ha="center", va="bottom", fontsize=9)
    fig.suptitle("Ключевые KPI портфеля (последний срез)", fontsize=12, y=1.02)
    plt.tight_layout()
    mpl_save_show(fig, stem)


def plot_segment_profile(raw: pd.DataFrame, stem: str = "segment_profile") -> None:
    plot_df = raw.copy()
    seg_order = [s for s in SEGMENT_ORDER if s in set(plot_df["segment"].astype(str))]
    seg_order += [s for s in plot_df["segment"].astype(str).unique().tolist() if s not in seg_order]
    plot_df["_seg"] = pd.Categorical(plot_df["segment"].astype(str), categories=seg_order, ordered=True)
    plot_df = plot_df.sort_values("_seg")
    dr = plot_df["default_rate"].to_numpy(dtype=float)
    norm = plt.Normalize(dr.min(), dr.max() + 1e-9)
    cmap = plt.cm.Reds
    colors = cmap(norm(dr))
    fig, ax = plt.subplots(figsize=(10.5, 5))
    xpos = np.arange(len(plot_df))
    ax.bar(xpos, plot_df["n_contracts"], color=colors, edgecolor="white")
    ax.set_xticks(xpos)
    ax.set_xticklabels(plot_df["segment"].astype(str))
    ax.set_xlabel("Сегмент")
    ax.set_ylabel("Договоров")
    ax.set_title("Число договоров по сегментам (последний срез)")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    fig.colorbar(sm, ax=ax, label="Default rate")
    for i, row in plot_df.reset_index(drop=True).iterrows():
        ax.text(i, row["n_contracts"], str(int(row["n_contracts"])), ha="center", va="bottom", fontsize=8)
    plt.tight_layout()
    mpl_save_show(fig, stem)


def woe_iv_table(df: pd.DataFrame, feature: str, target_col: str, bins: int = 10) -> pd.DataFrame:
    s = df[[feature, target_col]].copy().dropna()
    if s[feature].dtype.kind in "biufc":
        try:
            s["bin"] = pd.qcut(s[feature], q=bins, duplicates="drop")
        except Exception:
            s["bin"] = pd.cut(s[feature], bins=bins)
    else:
        s["bin"] = s[feature].astype(str)
    g = s.groupby("bin", observed=False)[target_col].agg(["sum", "count"])
    g.columns = ["bad", "total"]
    g["good"] = g["total"] - g["bad"]
    good_total, bad_total = g["good"].sum(), g["bad"].sum()
    g["dist_good"] = g["good"] / (good_total + 1e-9)
    g["dist_bad"] = g["bad"] / (bad_total + 1e-9)
    g["woe"] = np.log((g["dist_good"] + 1e-9) / (g["dist_bad"] + 1e-9))
    g["iv_part"] = (g["dist_good"] - g["dist_bad"]) * g["woe"]
    g["iv"] = g["iv_part"].sum()
    return g.reset_index()


def information_value_ranking(df: pd.DataFrame, features: list, target_col: str, bins: int = 10) -> pd.DataFrame:
    rows = []
    for f in features:
        try:
            iv = float(woe_iv_table(df, f, target_col, bins=bins)["iv"].iloc[0])
        except Exception:
            iv = np.nan
        rows.append({"feature": f, "iv": iv})
    out = pd.DataFrame(rows).dropna().sort_values("iv", ascending=False).reset_index(drop=True)
    return out


def plot_iv_ranking(iv_tbl: pd.DataFrame, top: int = 15, stem: str = "iv_ranking") -> None:
    dfp = iv_tbl.head(top).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, max(4.5, top * 0.28)))
    sns.barplot(data=dfp, x="iv", y="feature", orient="h", color="#3b6ea5", ax=ax)
    ax.set_title(f"Информационная ценность (топ-{top})")
    ax.set_xlabel("IV")
    plt.tight_layout()
    mpl_save_show(fig, stem)


def plot_feature_importance(fi: pd.DataFrame, top: int = 20, stem: str = "feature_importance") -> None:
    sub = fi.head(top).iloc[::-1]
    fig, ax = plt.subplots(figsize=(10, max(5, top * 0.26)))
    sns.barplot(data=sub, x="importance", y="feature", orient="h", color="#2a9d8f", ax=ax)
    ax.set_title(f"Важность признаков (топ-{top}, gain LightGBM)")
    ax.set_xlabel("Gain")
    plt.tight_layout()
    mpl_save_show(fig, stem)


def _pct_ru(value: float, decimals: int = 2) -> str:
    s = f"{value * 100:.{decimals}f}"
    return f"{s.replace('.', ',')}%"


def plot_model_metrics(train_m: dict, test_m: dict, cv_mean: float, cv_std: float, stem: str = "model_metrics") -> None:
    keys = ["roc_auc", "pr_auc", "gini", "ks", "f1@thr"]
    labels_k = ["ROC-AUC", "PR-AUC", "Gini", "KS", "F1 @ thr"]
    y_train = [float(train_m[k]) for k in keys]
    y_test = [float(test_m[k]) for k in keys]
    x = np.arange(len(labels_k))
    w = 0.36
    fig, ax = plt.subplots(figsize=(11, 5.5))
    ax.bar(x - w / 2, y_train, width=w, label="Train", color="#1a6fb0", edgecolor="white")
    ax.bar(x + w / 2, y_test, width=w, label="Test", color="#c45c26", edgecolor="white")
    ax.set_xticks(x)
    ax.set_xticklabels(labels_k)
    ax.set_ylabel("Доля / коэффициент")
    ax.set_ylim(0, 1.12)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
    ax.grid(axis="y", alpha=0.25)
    ax.set_facecolor("#fafafa")
    for i, (vt, ve) in enumerate(zip(y_train, y_test)):
        ax.text(i - w / 2, vt, _pct_ru(vt), ha="center", va="bottom", fontsize=8, color="#0d3d61")
        ax.text(i + w / 2, ve, _pct_ru(ve), ha="center", va="bottom", fontsize=8, color="#6b3418")
    cv_mu = f"{cv_mean:.4f}".replace(".", ",")
    cv_sg = f"{cv_std:.4f}".replace(".", ",")
    ax.set_title("Метрики модели (train / test)")
    ax.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.14), framealpha=0.9)
    fig.text(0.5, 0.02, f"CV ROC-AUC: {cv_mu} ± {cv_sg}", ha="center", fontsize=10, color="#334155")
    plt.tight_layout(rect=(0, 0.06, 1, 1))
    mpl_save_show(fig, stem)


def plot_ews_zones(summary: pd.DataFrame, stem: str = "ews_zones") -> None:
    order = ["green", "yellow", "red"]
    sdf = summary.copy()
    sdf["zone"] = pd.Categorical(sdf["zone"], categories=order, ordered=True)
    sdf = sdf.sort_values("zone")
    labels = sdf["zone"].astype(str).map(ZONE_DISPLAY)
    colors = [ZONE_COLORS[str(z)] for z in sdf["zone"]]
    fig, ax = plt.subplots(figsize=(9.5, 5))
    ax.bar(labels.astype(str), sdf["n_contracts"], color=colors, edgecolor="white")
    ax.set_xlabel("Зона")
    ax.set_ylabel("Число договоров")
    ax.set_title("Распределение договоров по зонам EWS")
    for i, v in enumerate(sdf["n_contracts"]):
        ax.text(i, v, str(int(v)), ha="center", va="bottom")
    plt.tight_layout()
    mpl_save_show(fig, stem)


def ks_statistic(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    order = np.argsort(-y_score)
    y_sorted = y_true[order]
    cum_pos = np.cumsum(y_sorted) / max(y_sorted.sum(), 1)
    cum_neg = np.cumsum(1 - y_sorted) / max((1 - y_sorted).sum(), 1)
    return float(np.max(np.abs(cum_pos - cum_neg)))


def gini_coef(y_true: np.ndarray, y_score: np.ndarray) -> float:
    return 2 * roc_auc_score(y_true, y_score) - 1


def binary_report(y_true: np.ndarray, y_score: np.ndarray, threshold: float = 0.5) -> dict:
    from sklearn.metrics import confusion_matrix, f1_score, brier_score_loss
    y_pred = (y_score >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "roc_auc": float(roc_auc_score(y_true, y_score)),
        "pr_auc": float(average_precision_score(y_true, y_score)),
        "gini": float(gini_coef(y_true, y_score)),
        "ks": float(ks_statistic(y_true, y_score)),
        "f1@thr": float(f1_score(y_true, y_pred, zero_division=0)),
        "brier": float(brier_score_loss(y_true, y_score)),
    }


def load_table():
    xlsx = Path("3_5.xlsx")
    pq = Path("3_5.parquet")
    if not xlsx.exists():
        raise FileNotFoundError("Сначала выполните ячейку с wget для 3_5.xlsx")
    if pq.exists():
        return pd.read_parquet(pq)
    df = pd.read_excel(xlsx, engine="openpyxl")
    try:
        df.to_parquet(pq, index=False)
    except Exception:
        pass
    return df


def cast_types(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for c in DATE_COLS:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")
    for c in NUMERIC_COLS:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def add_derived(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    ref = df.get("report_date_as_of", pd.Series(pd.Timestamp.today(), index=df.index))
    if "date_borrower_birthday" in df.columns:
        df["age"] = ((ref - df["date_borrower_birthday"]).dt.days / 365.25).round(1)
    if "date_issued" in df.columns:
        df["months_on_book"] = ((ref - df["date_issued"]).dt.days / 30.44).round(1)
    if "last_payment_date" in df.columns:
        df["days_since_last_payment"] = (ref - df["last_payment_date"]).dt.days
    if {"initial_amount", "available_limit"}.issubset(df.columns):
        df["utilization"] = (
            (df["initial_amount"] - df["available_limit"]) / (df["initial_amount"].abs() + 1e-6)
        ).clip(lower=0, upper=1.5)
    if {"payment_sum_1m", "payment_sum_2m"}.issubset(df.columns):
        df["payment_ratio_mom"] = df["payment_sum_1m"] / (df["payment_sum_2m"].abs() + 1e-6)
    if "bankruptcy_stage" in df.columns:
        df["is_bankrupt"] = (
            df["bankruptcy_stage"].notna() & (df["bankruptcy_stage"].astype(str) != "UNKNOWN")
        ).astype(np.int8)
    else:
        df["is_bankrupt"] = 0
    return df


def build_default_flag(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    dpd = pd.to_numeric(df.get("dpd"), errors="coerce").fillna(0)
    pdn = pd.to_numeric(df.get("pdn_current"), errors="coerce").fillna(0)
    quality = df.get("quality_category", pd.Series(index=df.index, dtype="string")).astype(str)
    is_bankrupt = df.get("is_bankrupt", pd.Series(0, index=df.index)).astype(int)
    df[TARGET_COL] = (
        (dpd > DEFAULT_DPD_THRESHOLD)
        | (pdn > PDN_DEFAULT_THRESHOLD)
        | (quality.isin(QUALITY_BAD))
        | (is_bankrupt == 1)
    ).astype(np.int8)
    return df


def take_last_slice(df: pd.DataFrame) -> pd.DataFrame:
    if "report_date_as_of" not in df.columns or "credit_id" not in df.columns:
        return df
    return df.sort_values(["credit_id", "report_date_as_of"]).drop_duplicates(subset=["credit_id"], keep="last")


def rule_based_segment(df: pd.DataFrame) -> pd.Series:
    seg = pd.Series("stable", index=df.index, dtype="object")
    util = df.get("utilization", pd.Series(0, index=df.index))
    dpd = df.get("dpd", pd.Series(0, index=df.index))
    pay_mom = df.get("payment_ratio_mom", pd.Series(1, index=df.index))
    dslp = df.get("days_since_last_payment", pd.Series(0, index=df.index))
    is_bankrupt = df.get("is_bankrupt", pd.Series(0, index=df.index))
    seg.loc[(util > 0.6) & (dpd <= 5)] = "growing"
    seg.loc[(pay_mom < 0.5) | (dslp > 45)] = "stress"
    seg.loc[(dpd > 30) | (is_bankrupt == 1)] = "delinquent"
    return seg


def segment_profile_table(df: pd.DataFrame, seg_col: str = "segment") -> pd.DataFrame:
    work = df
    if "credit_id" in df.columns and df["credit_id"].duplicated().any():
        work = df.sort_values(["report_date_as_of", "credit_id"], na_position="last").drop_duplicates(
            subset=["credit_id"], keep="last"
        )
    agg = {"credit_id": "nunique"}
    for c in ["total_debt", "available_limit", "payment_sum_1m"]:
        if c in work.columns:
            agg[c] = "sum"
    for c in ["pdn_current", "dpd", "utilization"]:
        if c in work.columns:
            agg[c] = "mean"
    if TARGET_COL in work.columns:
        agg[TARGET_COL] = "mean"
    out = work.groupby(seg_col, dropna=False).agg(agg).reset_index()
    out = out.rename(columns={"credit_id": "n_contracts", TARGET_COL: "default_rate"})
    return out.sort_values("n_contracts", ascending=False).reset_index(drop=True)


def portfolio_kpi_dict(df: pd.DataFrame) -> dict:
    return {
        "n_contracts": int(df["credit_id"].nunique()) if "credit_id" in df else len(df),
        "n_borrowers": int(df["borrower_id"].nunique()) if "borrower_id" in df else np.nan,
        "total_debt": float(df["total_debt"].sum()) if "total_debt" in df else np.nan,
        "available_limit": float(df["available_limit"].sum()) if "available_limit" in df else np.nan,
        "avg_pdn_current": float(df["pdn_current"].mean()) if "pdn_current" in df else np.nan,
        "avg_dpd": float(df["dpd"].mean()) if "dpd" in df else np.nan,
        "default_rate": float(df[TARGET_COL].mean()) if TARGET_COL in df else np.nan,
        "reserve_amount": float(df["total_debt_reserve"].sum()) if "total_debt_reserve" in df else np.nan,
    }


raw = load_table()
df = cast_types(raw)
if {"sold_flg", "forgiven_flg"}.issubset(df.columns):
    df = df[(df["sold_flg"] != 1) & (df["forgiven_flg"] != 1)].copy()
df = add_derived(df)
df = build_default_flag(df)
df = take_last_slice(df)
df["segment"] = rule_based_segment(df)

kpi = portfolio_kpi_dict(df)
seg_tbl = segment_profile_table(df)

print("Срез:", df.shape, "KPI default_rate:", round(kpi.get("default_rate", 0) or 0, 4))


In [ ]:
plot_portfolio_kpi(kpi)
plot_segment_profile(seg_tbl)


In [ ]:
from lightgbm import LGBMClassifier

ML_NUM = [
    "initial_amount", "total_debt", "available_limit", "dpd", "pdn_initial", "pdn_current",
    "utilization", "months_on_book", "age", "days_since_last_payment", "payment_ratio_mom",
    "payment_sum_1m", "payment_sum_2m", "payment_sum_3m", "cnt_all_payments", "psk_current",
    "reserve_rate", "ifrs_provision_rate",
]
ML_CAT = ["sex", "loan_program", "quality_category", "region_from_address", "bankruptcy_stage"]

num_f = [c for c in ML_NUM if c in df.columns]
cat_f = [c for c in ML_CAT if c in df.columns]

work = df.copy()
if len(work) > MAX_TRAIN_ROWS:
    work = work.sample(n=MAX_TRAIN_ROWS, random_state=RANDOM_STATE)

X = work[num_f + cat_f]
y = work[TARGET_COL].astype(int)

num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler(with_mean=False))])
try:
    cat_enc = OneHotEncoder(handle_unknown="ignore", min_frequency=0.01, sparse_output=True)
except TypeError:
    cat_enc = OneHotEncoder(handle_unknown="ignore", sparse=True)
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", cat_enc)])

if cat_f:
    pre = ColumnTransformer([("num", num_pipe, num_f), ("cat", cat_pipe, cat_f)])
else:
    pre = ColumnTransformer([("num", num_pipe, num_f)])

clf = LGBMClassifier(
    objective="binary", learning_rate=0.05, num_leaves=63, max_depth=-1,
    min_child_samples=50, reg_alpha=0.1, reg_lambda=0.1, n_estimators=400,
    class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
pipe = Pipeline([("prep", pre), ("clf", clf)])

if "report_date_as_of" in work.columns and work["report_date_as_of"].notna().any():
    order = np.argsort(work["report_date_as_of"].values)
    idx = np.arange(len(work))[order]
    split_i = int(len(idx) * (1 - TEST_SIZE))
    tr_idx, te_idx = idx[:split_i], idx[split_i:]
else:
    tr_idx, te_idx = train_test_split(np.arange(len(work)), test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)

X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
y_tr, y_te = y.iloc[tr_idx], y.iloc[te_idx]

pipe.fit(X_tr, y_tr)
p_tr = pipe.predict_proba(X_tr)[:, 1]
p_te = pipe.predict_proba(X_te)[:, 1]

train_m = binary_report(y_tr.values, p_tr)
test_m = binary_report(y_te.values, p_te)

cv_scores = []
if "report_date_as_of" in work.columns and work["report_date_as_of"].notna().sum() > 100:
    tss = TimeSeriesSplit(n_splits=3)
    splits = list(tss.split(X))
    for tr, va in splits:
        est = clone(pipe)
        est.fit(X.iloc[tr], y.iloc[tr])
        cv_scores.append(roc_auc_score(y.iloc[va], est.predict_proba(X.iloc[va])[:, 1]))
else:
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    splits = list(skf.split(X, y))
    for tr, va in splits:
        est = clone(pipe)
        est.fit(X.iloc[tr], y.iloc[tr])
        cv_scores.append(roc_auc_score(y.iloc[va], est.predict_proba(X.iloc[va])[:, 1]))

cv_scores = np.asarray(cv_scores, dtype=float)
cv_mean, cv_std = float(np.mean(cv_scores)), float(np.std(cv_scores))

plot_model_metrics(train_m, test_m, cv_mean, cv_std)

prep_f = pipe.named_steps["prep"].get_feature_names_out()
imp = pipe.named_steps["clf"].feature_importances_
fi_df = pd.DataFrame({"feature": prep_f, "importance": imp}).sort_values("importance", ascending=False)
plot_feature_importance(fi_df)

iv_feats = [c for c in num_f if c in work.columns and c != TARGET_COL]
iv_tbl = information_value_ranking(work[iv_feats + [TARGET_COL]], iv_feats, TARGET_COL)
plot_iv_ranking(iv_tbl)


In [ ]:
scores = pipe.predict_proba(X)[:, 1]
W = work.reset_index(drop=True)
rules_df = apply_rules(W, EWS_RULES)

s = scores.astype(float)
zone = np.where(s < EWS_ZONES["green"]["score_max"], "green", np.where(s < EWS_ZONES["yellow"]["score_max"], "yellow", "red"))
ews_scored = pd.DataFrame({
    "credit_id": W["credit_id"].values if "credit_id" in W.columns else np.arange(len(W)),
    "risk_score": scores,
    "total_debt": W["total_debt"].values if "total_debt" in W.columns else np.zeros(len(W)),
    "zone": zone,
})
ews_scored["rules_weight_sum"] = rules_df["rules_weight_sum"].values

escalate = ews_scored["rules_weight_sum"] >= 3
ews_scored.loc[escalate & (ews_scored["zone"] == "green"), "zone"] = "yellow"
ews_scored.loc[ews_scored["rules_weight_sum"] >= 5, "zone"] = "red"

summary = (
    ews_scored.groupby("zone")
    .agg(n_contracts=("credit_id", "nunique"), sum_debt=("total_debt", "sum"), avg_score=("risk_score", "mean"))
    .reindex(["green", "yellow", "red"])
    .reset_index()
)
plot_ews_zones(summary)


In [ ]:
num_cols = [c for c in ["dpd", "pdn_current", "total_debt", "utilization"] if c in df.columns]
if len(num_cols) >= 2:
    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(df[num_cols].corr(numeric_only=True), annot=True, fmt=".2f", cmap="RdYlBu_r", center=0, ax=ax)
    ax.set_title("Корреляция (доп. к графикам ВКР)")
    plt.tight_layout()
    mpl_save_show(fig, "correlation")
